# BLS API Live Tester

Manually verify live connectivity to the BLS Public Data API using the client code from this repo.

- **v1** — unauthenticated, no key required
- **v2** — uses `BLS_API_KEY` from `.env`, automatically skips if key is absent

Series tested: **CUUR0000SA0** (CPI-U, All Items, U.S. City Average)  
BLS reference: https://data.bls.gov/timeseries/CUUR0000SA0

Run cells top to bottom. Each cell prints its own output.

## Imports

In [2]:
import sys
import os

import pandas as pd
from pydantic import SecretStr

# Add src/ to the path so example_server can be imported
sys.path.insert(0, os.path.join(os.path.abspath("."), "src"))

# config.py — settings, URL constants, limits, version selector
import example_server.config as cfg
from example_server.config import (
    BLS_V1_BASE_URL,
    BLS_V2_BASE_URL,
    BLS_V1_MAX_SERIES,
    BLS_V1_MAX_YEAR_RANGE,
    BLS_V1_DAILY_LIMIT,
    BLS_V2_MAX_SERIES,
    BLS_V2_MAX_YEAR_RANGE,
    BLS_V2_DAILY_LIMIT,
    get_bls_api_version,
    settings,
)

# utils.py — HTTP client and response unwrapper
from example_server.utils import (
    fetch_bls,
    unwrap_bls_response,
    BLSAPIError,
    UpstreamServiceError,
)

print("All imports OK")

All imports OK


## Configuration check

Shows which API version will be used and what limits apply.

In [3]:
key_configured = bool(settings.bls_api_key.get_secret_value())

print(f"BLS_API_KEY configured : {key_configured}")
print(f"API version selected   : {get_bls_api_version()}")
print()
print(f"v1 endpoint  : {BLS_V1_BASE_URL}")
print(f"v1 limits    : {BLS_V1_MAX_SERIES} series/request,  {BLS_V1_MAX_YEAR_RANGE}-year range,  {BLS_V1_DAILY_LIMIT} requests/day")
print()
print(f"v2 endpoint  : {BLS_V2_BASE_URL}")
print(f"v2 limits    : {BLS_V2_MAX_SERIES} series/request,  {BLS_V2_MAX_YEAR_RANGE}-year range,  {BLS_V2_DAILY_LIMIT} requests/day")

BLS_API_KEY configured : True
API version selected   : v2

v1 endpoint  : https://api.bls.gov/publicAPI/v1/
v1 limits    : 25 series/request,  10-year range,  25 requests/day

v2 endpoint  : https://api.bls.gov/publicAPI/v2/
v2 limits    : 50 series/request,  20-year range,  500 requests/day


## Test 1 — BLS API v1 (unauthenticated)

Clears the key temporarily so v1 is used regardless of what is in `.env`.

In [4]:
# Temporarily clear the key so this cell always hits v1
saved_key = cfg.settings.bls_api_key
cfg.settings.bls_api_key = SecretStr("")

print(f"API version : {get_bls_api_version()}")
print(f"Series      : CUUR0000SA0")
print(f"Years       : 2024 - 2025")
print()

raw_v1 = await fetch_bls({
    "seriesid":  ["CUUR0000SA0"],
    "startyear": "2024",
    "endyear":   "2025",
})

print(f"HTTP status  : 200 (no exception raised)")
print(f"BLS status   : {raw_v1['status']}")
print(f"BLS messages : {raw_v1.get('message') or 'none'}")

# Restore the key for the v2 cell
cfg.settings.bls_api_key = saved_key

API version : v1
Series      : CUUR0000SA0
Years       : 2024 - 2025

HTTP status  : 200 (no exception raised)
BLS status   : REQUEST_SUCCEEDED
BLS messages : none


In [5]:
# Unwrap the response and display as a DataFrame
series_v1 = unwrap_bls_response(raw_v1)[0]

rows_v1 = [
    {
        "series_id":   series_v1["seriesID"],
        "year":        obs["year"],
        "period":      obs["period"],
        "period_name": obs["periodName"],
        "value":       obs["value"],
        "footnotes":   ", ".join(f["code"] for f in obs["footnotes"] if f.get("code")),
    }
    for obs in series_v1["data"]
]

df_v1 = pd.DataFrame(rows_v1).sort_values(["year", "period"], ascending=[False, False]).reset_index(drop=True)

print(f"Observations returned: {len(df_v1)}")
print()
df_v1

Observations returned: 24



,series_id,year,period,period_name,value,footnotes
0,CUUR0000SA0,2025,M12,December,324.054,
1,CUUR0000SA0,2025,M11,November,324.122,
2,CUUR0000SA0,2025,M10,October,-,X
3,CUUR0000SA0,2025,M09,September,324.800,
4,CUUR0000SA0,2025,M08,August,323.976,
5,CUUR0000SA0,2025,M07,July,323.048,
6,CUUR0000SA0,2025,M06,June,322.561,
7,CUUR0000SA0,2025,M05,May,321.465,
8,CUUR0000SA0,2025,M04,April,320.795,
9,CUUR0000SA0,2025,M03,March,319.799,


## Test 2 — BLS API v2 (authenticated)

Uses the `BLS_API_KEY` from `.env`. The key is injected inside `fetch_bls()` in `utils.py`  
and never appears here. This cell skips automatically if no key is configured.

In [6]:
if not bool(settings.bls_api_key.get_secret_value()):
    print("SKIPPED — no BLS_API_KEY in .env")
    print("Add BLS_API_KEY=<your_key> to .env and re-run this cell.")
    raw_v2 = None
else:
    print(f"API version : {get_bls_api_version()}")
    print(f"Series      : CUUR0000SA0")
    print(f"Years       : 2024 - 2025")
    print(f"Key         : [REDACTED]")
    print()

    raw_v2 = await fetch_bls({
        "seriesid":  ["CUUR0000SA0"],
        "startyear": "2024",
        "endyear":   "2025",
    })

    print(f"HTTP status  : 200 (no exception raised)")
    print(f"BLS status   : {raw_v2['status']}")
    print(f"BLS messages : {raw_v2.get('message') or 'none'}")

API version : v2
Series      : CUUR0000SA0
Years       : 2024 - 2025
Key         : [REDACTED]

HTTP status  : 200 (no exception raised)
BLS status   : REQUEST_SUCCEEDED
BLS messages : none


In [8]:
if raw_v2 is None:
    print("No data — v2 was skipped above.")
else:
    series_v2 = unwrap_bls_response(raw_v2)[0]

    rows_v2 = [
        {
            "series_id":   series_v2["seriesID"],
            "year":        obs["year"],
            "period":      obs["period"],
            "period_name": obs["periodName"],
            "value":       obs["value"],
            "footnotes":   ", ".join(f["code"] for f in obs["footnotes"] if f.get("code")),
        }
        for obs in series_v2["data"]
    ]

    df_v2 = pd.DataFrame(rows_v2).sort_values(["year", "period"], ascending=[False, False]).reset_index(drop=True)

    print(f"Observations returned: {len(df_v2)}")
    print()
    print(df_v2)

Observations returned: 24

      series_id  year period period_name    value footnotes
0   CUUR0000SA0  2025    M12    December  324.054          
1   CUUR0000SA0  2025    M11    November  324.122          
2   CUUR0000SA0  2025    M10     October        -         X
3   CUUR0000SA0  2025    M09   September  324.800          
4   CUUR0000SA0  2025    M08      August  323.976          
5   CUUR0000SA0  2025    M07        July  323.048          
6   CUUR0000SA0  2025    M06        June  322.561          
7   CUUR0000SA0  2025    M05         May  321.465          
8   CUUR0000SA0  2025    M04       April  320.795          
9   CUUR0000SA0  2025    M03       March  319.799          
10  CUUR0000SA0  2025    M02    February  319.082          
11  CUUR0000SA0  2025    M01     January  317.671          
12  CUUR0000SA0  2024    M12    December  315.605          
13  CUUR0000SA0  2024    M11    November  315.493          
14  CUUR0000SA0  2024    M10     October  315.664          
15  CUUR0000S

## Compare v1 vs v2

Merges both DataFrames on year + period and checks that every `value` is identical.  
Both API versions serve the same underlying BLS data, so all rows should match.

In [7]:
if raw_v2 is None:
    print("Cannot compare — v2 was skipped.")
else:
    merged = df_v1[["year", "period", "period_name", "value"]].merge(
        df_v2[["year", "period", "value"]],
        on=["year", "period"],
        suffixes=("_v1", "_v2"),
    )

    merged["values_match"] = merged["value_v1"] == merged["value_v2"]

    all_match = merged["values_match"].all()
    print(f"Rows compared : {len(merged)}")
    print(f"All values match : {all_match}")
    print()
    merged

Rows compared : 24
All values match : True



## Bonus — multiple series with v2

v2 supports up to 50 series in one request. Fetch three CPI sub-indexes at once and  
display them stacked in a single DataFrame.

| Series | Description |
|---|---|
| CUUR0000SA0 | CPI-U All Items |
| CUUR0000SAF | CPI-U Food |
| CUUR0000SAE | CPI-U Energy |

In [8]:
if not bool(settings.bls_api_key.get_secret_value()):
    print("SKIPPED — BLS_API_KEY required for multi-series v2 request.")
else:
    print(f"API version : {get_bls_api_version()}")
    print(f"Series      : CUUR0000SA0, CUUR0000SAF, CUUR0000SAE")
    print(f"Years       : 2025")
    print()

    raw_multi = await fetch_bls({
        "seriesid":  ["CUUR0000SA0", "CUUR0000SAF", "CUUR0000SAE"],
        "startyear": "2025",
        "endyear":   "2025",
    })

    print(f"BLS status : {raw_multi['status']}")
    print()

    all_series = unwrap_bls_response(raw_multi)

    rows_multi = [
        {
            "series_id":   s["seriesID"],
            "year":        obs["year"],
            "period":      obs["period"],
            "period_name": obs["periodName"],
            "value":       obs["value"],
        }
        for s in all_series
        for obs in s["data"]
    ]

    df_multi = pd.DataFrame(rows_multi).sort_values(
        ["series_id", "year", "period"], ascending=[True, False, False]
    ).reset_index(drop=True)

    print(f"Total observations : {len(df_multi)}")
    print()
    df_multi

API version : v2
Series      : CUUR0000SA0, CUUR0000SAF, CUUR0000SAE
Years       : 2025



BLS status : REQUEST_SUCCEEDED

Total observations : 36

